# NHA Hackathon — Problem Statement 02  
## Radiological Image Based Condition Detection & Report Correlation  


- All pipeline-stage functions are **empty** and contain only **multiline comments** describing what each function must do.
- A final **main runner section** wires the skeleton together, iterates through discovered cases/files/pages, fills the page-level data structure, and prepares final outputs at the end.

The final participant solution is expected to help adjudicators:
1. Read and describe medical images for condition-relevant features.
2. Check internal consistency of written reports.
3. Correlate image findings with report findings.
4. Check claimed condition / package / STG context.
5. Reason about stage-of-care and episode timeline.
6. Produce reviewer-friendly, assistive outputs.

In [ ]:
# =========================
# BASIC IMPORTS
# =========================

from __future__ import annotations

import os
import json
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


## Download the Dataset
We have provided a dedicated widget to download the hackathon datasets directly from the platform into this notebook environment.

### 1. Import the Widget

from databank_download_widget import DatabankDownloadWidget

### 2. Download the Databank
Select the cell below and run it.
Enter the Databank ID for the hackathon package.
Enter your email and password for the platform.
Click the **Download** button.

The widget will download and unzip the data right into your current directory. You can monitor the progress in the status output area below the button.

### **Databank ID for PS2: 841403b5-bb4b-4e2a-a73f-570c1b1af8fb**

In [ ]:
from databank_download_widget import DatabankDownloadWidget

# Create an instance of the databank widget
databank_downloader = DatabankDownloadWidget()

# Display the widget
databank_downloader.display()

In [ ]:
# =========================
# CONFIGURATION
# =========================

INPUT_ROOT = Path("./Claims")
OUTPUT_ROOT = Path("./outputs")

SUPPORTED_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
SUPPORTED_PDF_EXTS = {".pdf"}
SUPPORTED_TEXT_REPORT_EXTS = {".txt", ".md", ".json"}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:

# =========================
# PACKAGE SCHEMAS / OUTPUT TEMPLATES
# =========================

PACKAGE_FOLDER_TO_CODE = {
    "MC011A": "MC011A",
    "MG029A": "MG029A",
    "SG039": "SG039C",
    "SG039C": "SG039C",
    "SU007A": "SU007A",
}

_DEFAULT = "Not assessed / Not seen"

PACKAGE_OUTPUT_FIELDS: Dict[str, List[str]] = {
    "MC011A": [
        "Claim No. ",
        "Package Code",
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)",
        "LMCA",
        "LAD",
        "LCX ",
        "RCA",
        "LMCA_Post",
        "LAD_Post",
        "LCX_Post",
        "RCA_Post",
        "Stent Visualization",
        "Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Inter",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
        "Reasoning_Post_Inter",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reasoning_Alignment",
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag for one month older documents for pre-procedure investigations \n(Yes/ No)",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        "Comment on image quality/ usability",
        "Summary PDF",
        "Remarks",
    ],
    "MG029A": [
        "Claim No. ",
        "Package_code",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)",
        "Lung fields",
        "Left and right CP angles",
        "Left and right hilum",
        "Midline shift",
        "Cardiac size",
        "Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
        "Reasoning_Inter",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reason for Non alignment",
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG \n(to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        "Comment on image quality/ usability",
        "Summary PDF",
    ],
    "SG039C": [
        "Claim No. ",
        "Package Code",
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
        "Liver",
        "Gall bladder",
        "Spleen",
        "Kidneys",
        "Urinary bladder",
        "Prostate/ Uterus",
        "Peritoneal fluid",
        "Intra:Within report consistency check (Consistent/Partially supported/Unsupportred",
        "Reasoning_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
        "Reason for Partially supported/Unsupported",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reasoning_Alignment",
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Pre-procedure investigation date\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag for one month older documents for pre-procedure investigations ",
        "Comment on image quality/ usability",
        "Summary PDF",
    ],
    "SU007A": [
        "Claim No. ",
        "Package code",
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)",
        "Pelvicalyceal system",
        "Ureter",
        "Urinary bladder",
        "Stone visualisation",
        "Any radio opaque shadow or calcification in abdomen region",
        "DJ stent/ PCN tube visualised",
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Intra",
        "Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)",
        "Reasoning_Pre_Inter",
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post",
        "Reasoning_Post_Intra",
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
        "Reasoning_Post_Inter",
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
        "Reasoning_Alignment",
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
        "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        "Admission Date\n(DD/MM/YYYY)",
        "Pre-procedure investigation date\n(DD/MM/YYYY)",
        "Post Procedure Investigation Date\n(DD/MM/YYYY)",
        "Discharge Date\n(DD/MM/YYYY)",
        "Flag for one month older documents for pre-procedure investigations ",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        "Comment on image quality/ usability",
        "Summary PDF",
        "Remarks",
    ],
}

PACKAGE_PLACEHOLDERS: Dict[str, Dict[str, Any]] = {
    "MC011A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Absent",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Absent",
        "LMCA": _DEFAULT,
        "LAD": _DEFAULT,
        "LCX ": _DEFAULT,
        "RCA": _DEFAULT,
        "LMCA_Post": _DEFAULT,
        "LAD_Post": _DEFAULT,
        "LCX_Post": _DEFAULT,
        "RCA_Post": _DEFAULT,
        "Stent Visualization": _DEFAULT,
        "Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Inter": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post": _DEFAULT,
        "Reasoning_Post_Inter": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reasoning_Alignment": _DEFAULT,
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG (to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)": "",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag for one month older documents for pre-procedure investigations \n(Yes/ No)": "No",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
        "Remarks": "",
    },
    "MG029A": {
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Absent",
        "Lung fields": _DEFAULT,
        "Left and right CP angles": _DEFAULT,
        "Left and right hilum": _DEFAULT,
        "Midline shift": _DEFAULT,
        "Cardiac size": _DEFAULT,
        "Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Inter": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reason for Non alignment": _DEFAULT,
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG \n(to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
    },
    "SG039C": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Absent",
        "Liver": _DEFAULT,
        "Gall bladder": _DEFAULT,
        "Spleen": _DEFAULT,
        "Kidneys": _DEFAULT,
        "Urinary bladder": _DEFAULT,
        "Prostate/ Uterus": _DEFAULT,
        "Peritoneal fluid": _DEFAULT,
        "Intra:Within report consistency check (Consistent/Partially supported/Unsupportred": _DEFAULT,
        "Reasoning_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for Partially supported/Unsupported": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reasoning_Alignment": _DEFAULT,
        "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG (to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Pre-procedure investigation date\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag for one month older documents for pre-procedure investigations ": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
    },
    "SU007A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Absent",
        "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)": "Absent",
        "Pelvicalyceal system": _DEFAULT,
        "Ureter": _DEFAULT,
        "Urinary bladder": _DEFAULT,
        "Stone visualisation": _DEFAULT,
        "Any radio opaque shadow or calcification in abdomen region": _DEFAULT,
        "DJ stent/ PCN tube visualised": _DEFAULT,
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reasoning_Pre_Inter": _DEFAULT,
        "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post": _DEFAULT,
        "Reasoning_Post_Intra": _DEFAULT,
        "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post": _DEFAULT,
        "Reasoning_Post_Inter": _DEFAULT,
        "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)": _DEFAULT,
        "Reasoning_Alignment": _DEFAULT,
        "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)": _DEFAULT,
        "Reason for deviation from STG (to be mapped to PPD and CPD section)": _DEFAULT,
        "Admission Date\n(DD/MM/YYYY)": "",
        "Pre-procedure investigation date\n(DD/MM/YYYY)": "",
        "Post Procedure Investigation Date\n(DD/MM/YYYY)": "",
        "Discharge Date\n(DD/MM/YYYY)": "",
        "Flag for one month older documents for pre-procedure investigations ": "No",
        "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)": "No",
        "Comment on image quality/ usability": _DEFAULT,
        "Summary PDF": "",
        "Remarks": "",
    },
}

PACKAGE_IMAGE_TYPE_DEFAULTS = {
    "MC011A": {"pre": "Cardiac angiogram (multiple stills)", "post": "Cardiac angiogram (multiple stills)"},
    "MG029A": {"post": "Chest X ray"},
    "SG039C": {"pre": "USG abdomen"},
    "SU007A": {"pre": "IVP", "post": "X ray KUB"},
}

PACKAGE_SECTION_MAP: Dict[str, List[Tuple[str, List[str]]]] = {
    "MC011A": [
        ("Claim details", ["Claim No. ", "Package Code"]),
        ("1: Image classification", [
            "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
            "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)",
        ]),
        ("2a: Pre-procedure Image description", ["LMCA", "LAD", "LCX ", "RCA"]),
        ("2b: Post-procedure Image description", ["LMCA_Post", "LAD_Post", "LCX_Post", "RCA_Post", "Stent Visualization"]),
        ("3a: Pre-procedure validity of reports", [
            "Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Inter",
        ]),
        ("3a: Post-procedure validity of reports (only inter)", [
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
            "Reasoning_Post_Inter",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reasoning_Alignment",
            "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
            "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag for one month older documents for pre-procedure investigations \n(Yes/ No)",
            "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
    "MG029A": [
        ("Claim details", ["Claim No. ", "Package_code"]),
        ("1: Image classification", ["Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"]),
        ("2b: Post Procedure Image description", ["Lung fields", "Left and right CP angles", "Left and right hilum", "Midline shift", "Cardiac size"]),
        ("3b: Post-procedure validity of reports", [
            "Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
            "Reasoning_Inter",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reason for Non alignment",
            "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG \n(to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
    "SG039C": [
        ("Claim details", ["Claim No. ", "Package Code"]),
        ("1: Image classification", ["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"]),
        ("2a: Pre-procedure Image description", ["Liver", "Gall bladder", "Spleen", "Kidneys", "Urinary bladder", "Prostate/ Uterus", "Peritoneal fluid"]),
        ("3a: Pre-procedure validity of reports", [
            "Intra:Within report consistency check (Consistent/Partially supported/Unsupportred",
            "Reasoning_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)",
            "Reason for Partially supported/Unsupported",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reasoning_Alignment",
            "Investigation and STG Alignment (Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Pre-procedure investigation date\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag for one month older documents for pre-procedure investigations ",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
    "SU007A": [
        ("Claim details", ["Claim No. ", "Package code"]),
        ("1: Image classification", [
            "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)",
            "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)",
        ]),
        ("2a: Pre- Procedure Image description", ["Pelvicalyceal system", "Ureter", "Urinary bladder", "Stone visualisation"]),
        ("2b: Post Procedure Image description", ["Any radio opaque shadow or calcification in abdomen region", "DJ stent/ PCN tube visualised"]),
        ("3a: Pre-procedure validity of reports", [
            "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Intra",
            "Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)",
            "Reasoning_Pre_Inter",
        ]),
        ("3b: Post-procedure validity of reports", [
            "Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post",
            "Reasoning_Post_Intra",
            "Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post",
            "Reasoning_Post_Inter",
        ]),
        ("3: Procedure code and STG alignment", [
            "Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)",
            "Reasoning_Alignment",
            "Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)",
            "Reason for deviation from STG (to be mapped to PPD and CPD section)",
        ]),
        ("4: Stage awareness", [
            "Admission Date\n(DD/MM/YYYY)",
            "Pre-procedure investigation date\n(DD/MM/YYYY)",
            "Post Procedure Investigation Date\n(DD/MM/YYYY)",
            "Discharge Date\n(DD/MM/YYYY)",
            "Flag for one month older documents for pre-procedure investigations ",
            "Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)",
        ]),
        ("5. Quality", ["Comment on image quality/ usability"]),
    ],
}

PACKAGE_TRAILING_FIELDS: Dict[str, List[str]] = {
    "MC011A": ["Summary PDF", "Remarks"],
    "MG029A": ["Summary PDF"],
    "SG039C": ["Summary PDF"],
    "SU007A": ["Summary PDF", "Remarks"],
}

PACKAGE_EXPORT_FIELD_RENAMES: Dict[str, Dict[str, str]] = {
    "MC011A": {
        "LMCA_Post": "LMCA",
        "LAD_Post": "LAD",
        "LCX_Post": "LCX",
        "RCA_Post": "RCA",
        "Reasoning_Pre_Intra": "Reasoning",
        "Reasoning_Pre_Inter": "Reasoning (2)",
        "Reasoning_Post_Inter": "Reasoning",
        "Reasoning_Alignment": "Reasoning",
    },
    "MG029A": {
        "Reasoning_Intra": "Reasoning",
        "Reasoning_Inter": "Reasoning (2)",
    },
    "SG039C": {
        "Reasoning_Intra": "Reasoning",
    },
    "SU007A": {
        "Reasoning_Pre_Intra": "Reasoning",
        "Reasoning_Pre_Inter": "Reasoning (2)",
        "Reasoning_Post_Intra": "Reasoning",
        "Reasoning_Post_Inter": "Reasoning (2)",
        "Reasoning_Alignment": "Reasoning",
    },
}

PACKAGE_IMAGE_SECTION_FIELD_LABELS: Dict[str, Dict[str, str]] = {
    "MC011A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Type of radiological image in pre-auth/ pre-procedure stage  (mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Type of radiological image in cclaim submission/ post-procedure stage  (mention absent if no image)",
    },
    "MG029A": {
        "Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)": "Type of radiological image in cclaim submission/ post-procedure stage  (mention absent if no image)",
    },
    "SG039C": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Type of radiological image in pre-auth/ pre-procedure stage  (mention absent if no image)",
    },
    "SU007A": {
        "Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)": "Type of radiological image in pre-auth/ pre-procedure stage  (mention absent if no image)",
        "Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)": "Type of radiological image in cclaim submission/ post-procedure stage  (mention absent if no image)",
    },
}

def initialize_package_row(package_code: str, claim_no: str) -> Dict[str, Any]:
    row = {field_name: "" for field_name in PACKAGE_OUTPUT_FIELDS[package_code]}
    row.update(PACKAGE_PLACEHOLDERS[package_code])

    if package_code == "MG029A":
        row["Package_code"] = package_code
        row["Claim No. "] = claim_no
    elif package_code == "SU007A":
        row["Package code"] = package_code
        row["Claim No. "] = claim_no
    else:
        row["Package Code"] = package_code
        row["Claim No. "] = claim_no
    return row


### Example Usage of NHAclient LLM Call

Supports following models:

- Ministral 3B
- Ministral 8B
- Nemotron Nano 30B
- Gemma 3 12B
- Gemma 3 4B

Can be used by Participants anywhere

In [ ]:
from nha_client import NHAclient
import base64


clientId=""
clientSecret=""


nc = NHAclient(clientId, clientSecret)


with open("Sample.jpg", "rb") as image_file:
    image_bytes = image_file.read()

image_base64 = base64.b64encode(image_bytes).decode("utf-8")
data_url = f"data:image/jpeg;base64,{image_base64}"

response = nc.completion(
    model="", #use one of the models
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text", "text": "What do you see"},
            ],
        }
    ],
    metadata={
            "problem_statement":2
        }
)

print(response)



In [ ]:

# =========================
# DATA STRUCTURES
# =========================

@dataclass
class DiscoveredFile:
    package_code: str
    case_id: str
    file_path: str
    file_name: str
    file_ext: str
    file_kind: str  # medical_image | pdf_document | written_report | other


@dataclass
class PageData:
    package_code: str
    case_id: str
    source_file: str
    file_kind: str
    page_number: int
    page_id: str

    image_path: Optional[str] = None
    extracted_text: str = ""
    report_text: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    image_quality_assessment: Dict[str, Any] = field(default_factory=dict)
    image_observations: Dict[str, Any] = field(default_factory=dict)
    report_observations: Dict[str, Any] = field(default_factory=dict)
    internal_report_consistency: Dict[str, Any] = field(default_factory=dict)
    image_report_correlation: Dict[str, Any] = field(default_factory=dict)
    package_stg_alignment: Dict[str, Any] = field(default_factory=dict)
    timeline_stage_assessment: Dict[str, Any] = field(default_factory=dict)
    reviewer_notes: Dict[str, Any] = field(default_factory=dict)


@dataclass
class CaseResult:
    package_code: str
    case_id: str
    pages: List[PageData] = field(default_factory=list)
    structured_output_row: Dict[str, Any] = field(default_factory=dict)
    textual_summary: str = ""
    reviewer_flags: List[Dict[str, Any]] = field(default_factory=list)
    final_case_summary: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RunOutputs:
    all_case_results: List[CaseResult] = field(default_factory=list)
    package_json_rows: Dict[str, List[Dict[str, Any]]] = field(default_factory=dict)
    final_reviewer_flags: List[Dict[str, Any]] = field(default_factory=list)
    final_case_summaries: List[Dict[str, Any]] = field(default_factory=list)
    written_files: Dict[str, Any] = field(default_factory=dict)


In [ ]:
# =========================
# DATA ONBOARDING HELPERS
# =========================

def classify_input_file(file_path: Path) -> str:
    """
    Decide what kind of input file this is.

    Intended responsibilities:
    - Inspect the file extension and assign a coarse input type.
    - Recognize image files that may be direct radiology films or image captures.
    - Recognize PDFs that may contain radiology films, written reports, or mixed pages.
    - Recognize text-like report files such as TXT, MD, or JSON sidecar reports.
    - Return one of the standardized kinds used by the rest of the pipeline.
    """
    pass


def discover_case_files(input_root: Path) -> List[DiscoveredFile]:
    """
    Discover all files under the input root and map them into a case-centric structure.

    Intended responsibilities:
    - Traverse case folders recursively.
    - Derive case identifiers from folder names or configured structure.
    - Register each discovered file with its case ID, name, extension, and file kind.
    - Preserve enough file metadata for downstream processing and reviewer traceability.
    - Return a flat list of discovered files that the main runner can iterate over.
    """
    pass


def split_file_into_pages(discovered_file: DiscoveredFile) -> List[Dict[str, Any]]:
    """
    Split a discovered input file into page-level or image-level work items.

    Intended responsibilities:
    - For direct image files, create a single page/image work item.
    - For PDFs, render or enumerate pages and create one work item per page.
    - For text report files, create at least one logical work item that can be attached
      to the case and later correlated with images.
    - Preserve source provenance such as file name, page number, and page identifier.
    - Return lightweight dictionaries that can be converted into PageData objects.
    """
    pass


In [ ]:
# =========================
# 1. PAGE-LEVEL INGESTION / EXTRACTION
# =========================

def initialize_page_data(page_stub: Dict[str, Any]) -> PageData:
    """
    Convert a discovered page/image stub into the standard PageData structure.

    Intended responsibilities:
    - Populate all mandatory identifiers such as case ID, source file, page number, and page ID.
    - Attach source image path or source reference for later model calls.
    - Initialize all analysis dictionaries so later pipeline stages can write into them.
    - Keep the structure stable and uniform regardless of input file type.
    """
    pass


def extract_text_for_page(page_data: PageData) -> str:
    """
    Extract text content associated with the current page or report unit.

    Intended responsibilities:
    - Run OCR for scanned radiology reports or image-embedded text.
    - Read text directly from machine-readable PDFs where possible.
    - Load text from text-based report files when present.
    - Return the extracted text so it can populate extracted_text and/or report_text.
    - Preserve enough fidelity for later consistency checks and correlation analysis.
    """
    pass


def attach_text_to_page_data(page_data: PageData, extracted_text: str) -> PageData:
    """
    Write extracted textual content into the appropriate PageData fields.

    Intended responsibilities:
    - Decide whether extracted text should populate general extracted_text,
      report_text, or both.
    - Support cases where a page contains both image evidence and report text.
    - Maintain a consistent representation for downstream report analysis.
    """
    pass


In [ ]:
# =========================
# 2. IMAGE QUALITY / UNCERTAINTY PIPELINE
# =========================

def assess_image_quality(page_data: PageData) -> Dict[str, Any]:
    """
    Assess the technical readability and uncertainty level of the medical image.

    Intended responsibilities:
    - Evaluate clarity issues such as blur, low contrast, overexposure, cropping,
      partial views, glare, rotation, or artifacts.
    - Distinguish between 'feature not seen' and 'image not assessable'.
    - Produce structured quality indicators and reviewer-friendly uncertainty notes.
    - Return a dictionary that can be stored in page_data.image_quality_assessment.
    """
    pass


def write_image_quality_to_page(page_data: PageData, quality_result: Dict[str, Any]) -> PageData:
    """
    Store image-quality assessment output into the PageData structure.

    Intended responsibilities:
    - Save technical quality indicators.
    - Save reviewer-facing notes about whether the image can be interpreted.
    - Preserve this evidence for later correlation and scoring.
    """
    pass


In [ ]:
# =========================
# 3. IMAGE INTERPRETATION PIPELINE
# =========================

def interpret_medical_image(page_data: PageData) -> Dict[str, Any]:
    """
    Read and describe condition-relevant visual features from the medical image.

    Intended responsibilities:
    - Analyze radiological images such as X-ray, ultrasound, CT, MRI, angiogram,
      urethrogram, and related modalities.
    - Produce observation-style outputs such as 'features consistent with ...',
      'not clearly seen', or 'no convincing evidence of ...'.
    - Focus on assistive image interpretation rather than diagnosis.
    - Explicitly acknowledge uncertainty when image quality is insufficient.
    - Return structured visual findings and plain-language observations.
    """
    pass


def write_image_observations_to_page(page_data: PageData, image_result: Dict[str, Any]) -> PageData:
    """
    Store image interpretation results into the PageData structure.

    Intended responsibilities:
    - Save structured visual findings.
    - Save plain-language image observations.
    - Preserve evidence needed for downstream image-to-report comparison.
    """
    pass


In [ ]:
# =========================
# 4. WRITTEN REPORT ANALYSIS PIPELINE
# =========================

def analyze_written_report(page_data: PageData) -> Dict[str, Any]:
    """
    Analyze the written radiology report content associated with the page or case.

    Intended responsibilities:
    - Extract the report's observation section, impression section, or equivalent.
    - Normalize report content into structured findings.
    - Identify what the report is claiming the image shows.
    - Preserve wording and medical context required for later internal consistency checks.
    """
    pass


def check_internal_report_consistency(page_data: PageData) -> Dict[str, Any]:
    """
    Check whether the written report is internally consistent.

    Intended responsibilities:
    - Compare detailed observations in the body of the report with the final
      impression or interpretation section.
    - Flag contradictions, unsupported conclusions, or missing logical links.
    - Distinguish strong contradictions from mild ambiguity.
    - Return a structured consistency assessment for reviewer attention.
    """
    pass


def write_report_analysis_to_page(
    page_data: PageData,
    report_result: Dict[str, Any],
    consistency_result: Dict[str, Any]
) -> PageData:
    """
    Store written-report analysis outputs into PageData.

    Intended responsibilities:
    - Save extracted report observations.
    - Save internal consistency findings and contradiction flags.
    - Preserve these results for later image-to-report correlation.
    """
    pass


In [ ]:
# =========================
# 5. IMAGE-TO-REPORT CORRELATION PIPELINE
# =========================

def correlate_image_with_report(page_data: PageData) -> Dict[str, Any]:
    """
    Compare what the image appears to show with what the written report claims.

    Intended responsibilities:
    - Determine whether image findings and report statements are consistent,
      partially supported, unsupported, or mismatched.
    - Separate true absence of support from low-confidence / poor-quality cases.
    - Produce concise reviewer notes explaining why the correlation status was assigned.
    - Return a structured correlation result with evidence-oriented explanations.
    """
    pass


def write_correlation_to_page(page_data: PageData, correlation_result: Dict[str, Any]) -> PageData:
    """
    Store image-to-report correlation results into PageData.

    Intended responsibilities:
    - Save structured correlation labels and reviewer-facing explanations.
    - Preserve evidence needed for case-level summaries and flags.
    """
    pass


In [ ]:
# =========================
# 6. PACKAGE / STG / TIMELINE PIPELINE
# =========================

def assess_package_and_stg_alignment(page_data: PageData) -> Dict[str, Any]:
    """
    Assess whether the image findings make sense in the claimed package / STG context.

    Intended responsibilities:
    - Contextualize radiology findings against the claimed condition or procedure.
    - Check whether the submitted image type and apparent findings are relevant
      to the claimed package or stage of care.
    - Flag images that appear unrelated, weakly relevant, or inconsistent with
      the expected STG-driven evidentiary context.
    - Remain assistive and avoid diagnosis or approval decisions.
    """
    pass


def assess_timeline_and_stage_of_care(page_data: PageData) -> Dict[str, Any]:
    """
    Assess whether the image aligns with the episode timeline and stage of care.

    Intended responsibilities:
    - Determine whether the image appears pre-treatment, post-treatment, intra-procedure,
      or temporally unclear.
    - Use dates, textual cues, metadata, implants, post-op changes, or report language
      to estimate stage of care.
    - Flag possible old images, reused images, unrelated episodes, or timing mismatches.
    - Return a structured temporal / stage-awareness assessment.
    """
    pass


def write_context_assessments_to_page(
    page_data: PageData,
    stg_result: Dict[str, Any],
    timeline_result: Dict[str, Any]
) -> PageData:
    """
    Store package/STG and timeline assessments into PageData.

    Intended responsibilities:
    - Save contextual alignment findings.
    - Save stage-of-care and temporal plausibility findings.
    - Preserve these results for case-level summaries and reviewer flags.
    """
    pass


In [ ]:

# =========================
# EXPORT / SERIALIZATION HELPERS
# =========================

def ensure_json_serializable(data: Any) -> Any:
    """
    Convert notebook output objects into JSON-serializable forms.

    Intended responsibilities:
    - Recursively convert dataclasses and Path-like objects.
    - Make final outputs safe to dump as JSON.
    - Preserve structure without changing semantic meaning.
    """
    if hasattr(data, "__dataclass_fields__"):
        return ensure_json_serializable(asdict(data))
    if isinstance(data, Path):
        return str(data)
    if isinstance(data, dict):
        return {str(k): ensure_json_serializable(v) for k, v in data.items()}
    if isinstance(data, list):
        return [ensure_json_serializable(v) for v in data]
    if isinstance(data, tuple):
        return [ensure_json_serializable(v) for v in data]
    return data


def _pick_first_nonempty(*values: Any, default: str = "") -> str:
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and value.strip():
            return value.strip()
        if not isinstance(value, str):
            return str(value)
    return default


def _normalize_date_string(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    try:
        if pd is not None and hasattr(pd, "Timestamp") and isinstance(value, pd.Timestamp):
            return value.strftime("%d/%m/%Y")
    except Exception:
        pass
    try:
        return value.strftime("%d/%m/%Y")
    except Exception:
        return str(value)



def _build_textual_summary(case_result: CaseResult, row: Dict[str, Any]) -> str:
    summary_parts: List[str] = []
    pkg = case_result.package_code
    if pkg == "MC011A":
        summary_parts.append(f"Pre image: {row.get('Type of radiological image in pre-auth/ pre-procedure stage \\n(mention absent if no image)', 'Absent')}.")
        summary_parts.append(f"Post image: {row.get('Type of radiological image in cclaim submission/ post-procedure stage \\n(mention absent if no image)', 'Absent')}.")
        summary_parts.append(f"Pre report correlation: {row.get('Inter: Image vs Written Report Correlation \\n(Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
        summary_parts.append(f"Post report correlation: {row.get('Inter: Image vs Written Report Correlation \\n(Consistent/Partially supported/Unsupported)_Post', 'Not assessed / Not seen')}.")
    elif pkg == "MG029A":
        summary_parts.append(f"Post image: {row.get('Type of radiological image in cclaim submission/ post-procedure stage \\n(mention absent if no image)', 'Absent')}.")
        summary_parts.append(f"Lung fields: {row.get('Lung fields', 'Not assessed / Not seen')}.")
        summary_parts.append(f"Report correlation: {row.get('Inter: Image vs Written Report Correlation \\n(Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
    elif pkg == "SG039C":
        summary_parts.append(f"Pre image: {row.get('Type of radiological image in pre-auth/ pre-procedure stage \\n(mention absent if no image)', 'Absent')}.")
        summary_parts.append(f"Gall bladder: {row.get('Gall bladder', 'Not assessed / Not seen')}.")
        summary_parts.append(f"Report correlation: {row.get('Inter: Image vs Written Report Correlation \\n(Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
    elif pkg == "SU007A":
        summary_parts.append(f"Pre image: {row.get('Type of radiological image in pre-auth/ pre-procedure stage \\n(mention absent if no image)', 'Absent')}.")
        summary_parts.append(f"Post image: {row.get('Type of radiological image in cclaim submission/ post-procedure stage\\n(mention absent if no image)', 'Absent')}.")
        summary_parts.append(f"Stone visualisation: {row.get('Stone visualisation', 'Not assessed / Not seen')}.")
        summary_parts.append(f"Post device evidence: {row.get('DJ stent/ PCN tube visualised', 'Not assessed / Not seen')}.")
    if pkg == "MC011A":
        summary_parts.append(f"Package alignment: {row.get('Claim and Package Alignment \\n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)', 'Not assessed / Not seen')}.")
        summary_parts.append(f"STG alignment: {row.get('Investigation and STG Alignment (Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
    elif pkg == "MG029A":
        summary_parts.append(f"Package alignment: {row.get('Claim and Package Alignment \\n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)', 'Not assessed / Not seen')}.")
        summary_parts.append(f"STG alignment: {row.get('Investigation and STG Alignment \\n(Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
    elif pkg == "SG039C":
        summary_parts.append(f"Package alignment: {row.get('Claim and Package Alignment \\n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)', 'Not assessed / Not seen')}.")
        summary_parts.append(f"STG alignment: {row.get('Investigation and STG Alignment (Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
    elif pkg == "SU007A":
        summary_parts.append(f"Package alignment: {row.get('Claim and Package Alignment \\n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)', 'Not assessed / Not seen')}.")
        summary_parts.append(f"STG alignment: {row.get('Investigation and STG Alignment \\n(Consistent/Partially supported/Unsupported)', 'Not assessed / Not seen')}.")
    return " ".join(summary_parts).strip()



def _rename_field_for_export(package_code: str, field_name: str) -> str:
    return PACKAGE_EXPORT_FIELD_RENAMES.get(package_code, {}).get(field_name, field_name)


def _export_section_value(package_code: str, section_name: str, field_name: str, row: Dict[str, Any]) -> Any:
    display_name = _rename_field_for_export(package_code, field_name)

    if section_name == "1: Image classification":
        display_name = PACKAGE_IMAGE_SECTION_FIELD_LABELS.get(package_code, {}).get(field_name, display_name)
    elif section_name == "4: Stage awareness":
        display_name = display_name.replace("\n", " ")

    value = row.get(field_name, "")
    return display_name, value


def _row_to_sectioned_json(package_code: str, row: Dict[str, Any]) -> Dict[str, Any]:
    payload: Dict[str, Any] = {}
    for section_name, fields in PACKAGE_SECTION_MAP[package_code]:
        payload[section_name] = {}
        for field_name in fields:
            display_name, value = _export_section_value(package_code, section_name, field_name, row)
            payload[section_name][display_name] = value

    for trailing_field in PACKAGE_TRAILING_FIELDS.get(package_code, []):
        payload[trailing_field] = row.get(trailing_field, "")

    return payload


def build_package_json_row(case_result: CaseResult) -> Dict[str, Any]:
    package_code = case_result.package_code
    row = initialize_package_row(package_code, case_result.case_id)

    pages = case_result.pages
    image_pages = [p for p in pages if p.file_kind in {"medical_image", "pdf_document"}]
    report_pages = [p for p in pages if p.file_kind == "written_report"]

    page0 = pages[0] if pages else None
    image0 = image_pages[0] if image_pages else page0
    report0 = report_pages[0] if report_pages else page0

    quality_note = _pick_first_nonempty(
        (image0.image_quality_assessment or {}).get("quality_comment") if image0 else "",
        (image0.image_quality_assessment or {}).get("summary") if image0 else "",
        (image0.image_quality_assessment or {}).get("uncertainty_note") if image0 else "",
        "Not assessed / Not seen",
    )

    admission_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("admission_date") if page0 else "")
    pre_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("pre_investigation_date") if page0 else "")
    post_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("post_investigation_date") if page0 else "")
    discharge_date = _normalize_date_string((page0.timeline_stage_assessment or {}).get("discharge_date") if page0 else "")

    if package_code == "MC011A":
        if image_pages:
            row["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["MC011A"]["pre"]
            row["Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["MC011A"]["post"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["LMCA"] = _pick_first_nonempty(obs.get("pre_lmca"), default=row["LMCA"])
        row["LAD"] = _pick_first_nonempty(obs.get("pre_lad"), default=row["LAD"])
        row["LCX"] = _pick_first_nonempty(obs.get("pre_lcx"), default=row["LCX"])
        row["RCA"] = _pick_first_nonempty(obs.get("pre_rca"), default=row["RCA"])
        row["LMCA_Post"] = _pick_first_nonempty(obs.get("post_lmca"), default=row["LMCA_Post"])
        row["LAD_Post"] = _pick_first_nonempty(obs.get("post_lad"), default=row["LAD_Post"])
        row["LCX_Post"] = _pick_first_nonempty(obs.get("post_lcx"), default=row["LCX_Post"])
        row["RCA_Post"] = _pick_first_nonempty(obs.get("post_rca"), default=row["RCA_Post"])
        row["Stent Visualization"] = _pick_first_nonempty(obs.get("stent_visualization"), default=row["Stent Visualization"])

        row["Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(intra.get("pre_status"), intra.get("status"), default=row["Intra: Within report consistency check \n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Intra"] = _pick_first_nonempty(intra.get("pre_note"), intra.get("note"), default=row["Reasoning_Pre_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("pre_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Inter"] = _pick_first_nonempty(corr.get("pre_note"), corr.get("note"), default=row["Reasoning_Pre_Inter"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"] = _pick_first_nonempty(corr.get("post_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"])
        row["Reasoning_Post_Inter"] = _pick_first_nonempty(corr.get("post_note"), corr.get("note"), default=row["Reasoning_Post_Inter"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reasoning_Alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reasoning_Alignment"])
        row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG (to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG (to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Pre-procedure investigation date (From written report, or else the radiological image)\n(DD/MM/YYYY)"] = pre_date
        row["Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)"] = post_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag for one month older documents for pre-procedure investigations \n(Yes/ No)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("pre_older_than_one_month") if page0 else "", default=row["Flag for one month older documents for pre-procedure investigations \n(Yes/ No)"])
        row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("post_outside_admission_discharge_window") if page0 else "", default=row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"])
        row["Comment on image quality/ usability"] = quality_note
        row["Remarks"] = _pick_first_nonempty((image0.reviewer_notes or {}).get("remarks") if image0 else "", default=row["Remarks"])

    elif package_code == "MG029A":
        if image_pages:
            row["Type of radiological image in cclaim submission/ post-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["MG029A"]["post"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["Lung fields"] = _pick_first_nonempty(obs.get("lung_fields"), default=row["Lung fields"])
        row["Left and right CP angles"] = _pick_first_nonempty(obs.get("cp_angles"), default=row["Left and right CP angles"])
        row["Left and right hilum"] = _pick_first_nonempty(obs.get("hilum"), default=row["Left and right hilum"])
        row["Midline shift"] = _pick_first_nonempty(obs.get("midline_shift"), default=row["Midline shift"])
        row["Cardiac size"] = _pick_first_nonempty(obs.get("cardiac_size"), default=row["Cardiac size"])

        row["Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(intra.get("status"), default=row["Intra Report Consistency Check (within report)\n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Intra"] = _pick_first_nonempty(intra.get("note"), default=row["Reasoning_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Inter"] = _pick_first_nonempty(corr.get("note"), default=row["Reasoning_Inter"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reason for Non alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reason for Non alignment"])
        row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG \n(to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG \n(to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Post Procedure Investigation Date (From written report, or else the radiological image)\n(DD/MM/YYYY)"] = post_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("post_outside_admission_discharge_window") if page0 else "", default=row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"])
        row["Comment on image quality/ usability"] = quality_note

    elif package_code == "SG039C":
        if image_pages:
            row["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["SG039C"]["pre"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["Liver"] = _pick_first_nonempty(obs.get("liver"), default=row["Liver"])
        row["Gall bladder"] = _pick_first_nonempty(obs.get("gall_bladder"), default=row["Gall bladder"])
        row["Spleen"] = _pick_first_nonempty(obs.get("spleen"), default=row["Spleen"])
        row["Kidneys"] = _pick_first_nonempty(obs.get("kidneys"), default=row["Kidneys"])
        row["Urinary bladder"] = _pick_first_nonempty(obs.get("urinary_bladder"), default=row["Urinary bladder"])
        row["Prostate/ Uterus"] = _pick_first_nonempty(obs.get("prostate_or_uterus"), default=row["Prostate/ Uterus"])
        row["Peritoneal fluid"] = _pick_first_nonempty(obs.get("peritoneal_fluid"), default=row["Peritoneal fluid"])

        row["Intra:Within report consistency check (Consistent/Partially supported/Unsupportred"] = _pick_first_nonempty(intra.get("status"), default=row["Intra:Within report consistency check (Consistent/Partially supported/Unsupportred"])
        row["Reasoning_Intra"] = _pick_first_nonempty(intra.get("note"), default=row["Reasoning_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)"])
        row["Reason for Partially supported/Unsupported"] = _pick_first_nonempty(corr.get("note"), default=row["Reason for Partially supported/Unsupported"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reasoning_Alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reasoning_Alignment"])
        row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment (Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG (to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG (to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Pre-procedure investigation date\n(DD/MM/YYYY)"] = pre_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag for one month older documents for pre-procedure investigations "] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("pre_older_than_one_month") if page0 else "", default=row["Flag for one month older documents for pre-procedure investigations "])
        row["Comment on image quality/ usability"] = quality_note

    elif package_code == "SU007A":
        if image_pages:
            row["Type of radiological image in pre-auth/ pre-procedure stage \n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["SU007A"]["pre"]
            row["Type of radiological image in cclaim submission/ post-procedure stage\n(mention absent if no image)"] = PACKAGE_IMAGE_TYPE_DEFAULTS["SU007A"]["post"]
        obs = (image0.image_observations if image0 else {}) or {}
        corr = (image0.image_report_correlation if image0 else {}) or {}
        intra = (report0.internal_report_consistency if report0 else {}) or {}
        align = (image0.package_stg_alignment if image0 else {}) or {}

        row["Pelvicalyceal system"] = _pick_first_nonempty(obs.get("pcs"), default=row["Pelvicalyceal system"])
        row["Ureter"] = _pick_first_nonempty(obs.get("ureter"), default=row["Ureter"])
        row["Urinary bladder"] = _pick_first_nonempty(obs.get("urinary_bladder"), default=row["Urinary bladder"])
        row["Stone visualisation"] = _pick_first_nonempty(obs.get("stone_visualisation"), default=row["Stone visualisation"])
        row["Any radio opaque shadow or calcification in abdomen region"] = _pick_first_nonempty(obs.get("radio_opaque_shadow"), default=row["Any radio opaque shadow or calcification in abdomen region"])
        row["DJ stent/ PCN tube visualised"] = _pick_first_nonempty(obs.get("dj_stent_or_pcn_tube"), default=row["DJ stent/ PCN tube visualised"])

        row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(intra.get("pre_status"), intra.get("status"), default=row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Intra"] = _pick_first_nonempty(intra.get("pre_note"), intra.get("note"), default=row["Reasoning_Pre_Intra"])
        row["Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(corr.get("pre_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n (Consistent/Partially supported/Unsupported)"])
        row["Reasoning_Pre_Inter"] = _pick_first_nonempty(corr.get("pre_note"), corr.get("note"), default=row["Reasoning_Pre_Inter"])
        row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post"] = _pick_first_nonempty(intra.get("post_status"), intra.get("status"), default=row["Intra: Within report consistency check\n (Consistent/Partially supported/Unsupported)_Post"])
        row["Reasoning_Post_Intra"] = _pick_first_nonempty(intra.get("post_note"), intra.get("note"), default=row["Reasoning_Post_Intra"])
        row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"] = _pick_first_nonempty(corr.get("post_status"), corr.get("status"), default=row["Inter: Image vs Written Report Correlation \n(Consistent/Partially supported/Unsupported)_Post"])
        row["Reasoning_Post_Inter"] = _pick_first_nonempty(corr.get("post_note"), corr.get("note"), default=row["Reasoning_Post_Inter"])

        row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"] = _pick_first_nonempty(align.get("alignment_category"), default=row["Claim and Package Alignment \n(Both pre procedure and post procedure investigations align/ Only pre-procedure investigation aligns/ Only post-procedure investigtaion aligns/ None of the investigations align)"])
        row["Reasoning_Alignment"] = _pick_first_nonempty(align.get("alignment_reason"), align.get("note"), default=row["Reasoning_Alignment"])
        row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"] = _pick_first_nonempty(align.get("stg_alignment_outcome"), align.get("status"), default=row["Investigation and STG Alignment \n(Consistent/Partially supported/Unsupported)"])
        row["Reason for deviation from STG (to be mapped to PPD and CPD section)"] = _pick_first_nonempty(align.get("stg_reason"), align.get("note"), default=row["Reason for deviation from STG (to be mapped to PPD and CPD section)"])

        row["Admission Date\n(DD/MM/YYYY)"] = admission_date
        row["Pre-procedure investigation date\n(DD/MM/YYYY)"] = pre_date
        row["Post Procedure Investigation Date\n(DD/MM/YYYY)"] = post_date
        row["Discharge Date\n(DD/MM/YYYY)"] = discharge_date
        row["Flag for one month older documents for pre-procedure investigations "] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("pre_older_than_one_month") if page0 else "", default=row["Flag for one month older documents for pre-procedure investigations "])
        row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"] = _pick_first_nonempty((page0.timeline_stage_assessment or {}).get("post_outside_admission_discharge_window") if page0 else "", default=row["Flag before admssion date or after discharge date documents for post-procedure investigations \n(Yes - outside the window/\nNo - within the window)"])
        row["Comment on image quality/ usability"] = quality_note
        row["Remarks"] = _pick_first_nonempty((image0.reviewer_notes or {}).get("remarks") if image0 else "", default=row["Remarks"])

    row["Summary PDF"] = _build_textual_summary(case_result, row)
    return _row_to_sectioned_json(package_code, row)



def _extract_summary_pdf(structured_output: Dict[str, Any]) -> str:
    if not isinstance(structured_output, dict):
        return ""
    summary = structured_output.get("Summary PDF", "")
    return summary if isinstance(summary, str) else str(summary)


def write_run_outputs(run_outputs: RunOutputs, output_root: Path) -> None:
    """
    Write final notebook outputs to disk.

    Intended responsibilities:
    - Export package-specific structured rows into JSON files.
    - Keep one consolidated JSON containing all package outputs.
    - Preserve case-level summaries and reviewer flags for inspection.
    """
    output_root.mkdir(parents=True, exist_ok=True)

    package_dir = output_root / "package_json_outputs"
    package_dir.mkdir(parents=True, exist_ok=True)

    package_written_files: Dict[str, str] = {}
    consolidated_payload = {
        "package_outputs": {},
        "final_reviewer_flags": ensure_json_serializable(run_outputs.final_reviewer_flags),
        "final_case_summaries": ensure_json_serializable(run_outputs.final_case_summaries),
    }

    for package_code, rows in run_outputs.package_json_rows.items():
        file_path = package_dir / f"{package_code}_outputs.json"
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(ensure_json_serializable(rows), f, indent=2, ensure_ascii=False)
        package_written_files[package_code] = str(file_path)
        consolidated_payload["package_outputs"][package_code] = ensure_json_serializable(rows)

    consolidated_path = output_root / "ps2_all_package_outputs.json"
    with open(consolidated_path, "w", encoding="utf-8") as f:
        json.dump(consolidated_payload, f, indent=2, ensure_ascii=False)

    run_outputs.written_files = {
        "package_json_files": package_written_files,
        "consolidated_json": str(consolidated_path),
    }


In [ ]:

# =========================
# MAIN RUNNER
# =========================

def run_ps2_pipeline(input_root: Path) -> RunOutputs:
    """
    Main notebook runner that wires together all PS2 pipeline stages.

    Intended responsibilities:
    - Discover cases and files from the input root.
    - Group files by package and case.
    - Split each file into page/image work items.
    - Initialize the PageData structure for each work item.
    - Run all pipeline stages in sequence.
    - Aggregate page-level analysis into one package-specific structured
      output row per case.
    - Build final package-wise JSON outputs for export and display.
    """

    def _safe_call(fn, *args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            return {"_runner_error": f"{fn.__name__} failed: {exc}"}

    def _coerce_file_kind(file_path: Path) -> str:
        ext = file_path.suffix.lower()
        if ext in SUPPORTED_IMAGE_EXTS:
            return "medical_image"
        if ext in SUPPORTED_PDF_EXTS:
            return "pdf_document"
        if ext in SUPPORTED_TEXT_REPORT_EXTS:
            return "written_report"
        return "other"

    def _extract_package_code(root: Path, file_path: Path) -> str:
        try:
            rel = file_path.relative_to(root)
            top = rel.parts[0] if rel.parts else ""
        except Exception:
            top = file_path.parent.name
        return PACKAGE_FOLDER_TO_CODE.get(top, top)

    def _extract_case_id(root: Path, file_path: Path) -> str:
        try:
            rel = file_path.relative_to(root)
            rel_parts = rel.parts
        except Exception:
            rel_parts = (file_path.parent.name, file_path.name)

        if not rel_parts:
            return file_path.stem

        if len(rel_parts) >= 3:
            return "/".join(rel_parts[1:-1])
        if len(rel_parts) == 2:
            return file_path.stem
        return file_path.stem

    def _fallback_discover_case_files(root: Path) -> List[DiscoveredFile]:
        discovered: List[DiscoveredFile] = []
        if not root.exists():
            return discovered

        for file_path in sorted([p for p in root.rglob("*") if p.is_file()]):
            package_code = _extract_package_code(root, file_path)
            if package_code not in PACKAGE_OUTPUT_FIELDS:
                continue
            case_id = _extract_case_id(root, file_path)
            discovered.append(
                DiscoveredFile(
                    package_code=package_code,
                    case_id=case_id,
                    file_path=str(file_path),
                    file_name=file_path.name,
                    file_ext=file_path.suffix.lower(),
                    file_kind=_coerce_file_kind(file_path),
                )
            )
        return discovered

    def _fallback_split_file_into_pages(discovered_file: DiscoveredFile) -> List[Dict[str, Any]]:
        file_path = Path(discovered_file.file_path)
        if discovered_file.file_kind == "pdf_document":
            try:
                import fitz  # PyMuPDF
                doc = fitz.open(str(file_path))
                page_stubs = []
                for page_idx in range(len(doc)):
                    page_stubs.append(
                        {
                            "package_code": discovered_file.package_code,
                            "case_id": discovered_file.case_id,
                            "source_file": discovered_file.file_name,
                            "file_kind": discovered_file.file_kind,
                            "page_number": page_idx + 1,
                            "page_id": f"{discovered_file.package_code}::{discovered_file.case_id}::{discovered_file.file_name}::{page_idx + 1}",
                            "image_path": discovered_file.file_path,
                            "file_path": discovered_file.file_path,
                        }
                    )
                doc.close()
                return page_stubs or [{
                    "package_code": discovered_file.package_code,
                    "case_id": discovered_file.case_id,
                    "source_file": discovered_file.file_name,
                    "file_kind": discovered_file.file_kind,
                    "page_number": 1,
                    "page_id": f"{discovered_file.package_code}::{discovered_file.case_id}::{discovered_file.file_name}::1",
                    "image_path": discovered_file.file_path,
                    "file_path": discovered_file.file_path,
                }]
            except Exception:
                pass

        return [
            {
                "package_code": discovered_file.package_code,
                "case_id": discovered_file.case_id,
                "source_file": discovered_file.file_name,
                "file_kind": discovered_file.file_kind,
                "page_number": 1,
                "page_id": f"{discovered_file.package_code}::{discovered_file.case_id}::{discovered_file.file_name}::1",
                "image_path": discovered_file.file_path if discovered_file.file_kind in {"medical_image", "pdf_document"} else None,
                "file_path": discovered_file.file_path,
            }
        ]

    def _fallback_initialize_page_data(page_stub: Dict[str, Any]) -> PageData:
        return PageData(
            package_code=page_stub["package_code"],
            case_id=page_stub["case_id"],
            source_file=page_stub["source_file"],
            file_kind=page_stub["file_kind"],
            page_number=page_stub["page_number"],
            page_id=page_stub["page_id"],
            image_path=page_stub.get("image_path"),
            metadata={"file_path": page_stub.get("file_path", ""), "runner_initialized": True},
        )

    def _default_reviewer_flags(page_data: PageData) -> List[Dict[str, Any]]:
        flags: List[Dict[str, Any]] = []

        if (page_data.image_quality_assessment or {}).get("poor_quality"):
            flags.append({
                "package_code": page_data.package_code,
                "case_id": page_data.case_id,
                "page_id": page_data.page_id,
                "flag_type": "image_quality",
                "message": (page_data.image_quality_assessment or {}).get("uncertainty_note", "Image quality issue flagged."),
            })

        corr_status = (page_data.image_report_correlation or {}).get("status")
        if corr_status in {"unsupported", "mismatch", "inconsistent"}:
            flags.append({
                "package_code": page_data.package_code,
                "case_id": page_data.case_id,
                "page_id": page_data.page_id,
                "flag_type": "image_report_correlation",
                "message": (page_data.image_report_correlation or {}).get("note", "Image/report mismatch flagged."),
            })

        internal_status = (page_data.internal_report_consistency or {}).get("status")
        if internal_status in {"inconsistent", "contradiction"}:
            flags.append({
                "package_code": page_data.package_code,
                "case_id": page_data.case_id,
                "page_id": page_data.page_id,
                "flag_type": "internal_report_consistency",
                "message": (page_data.internal_report_consistency or {}).get("note", "Written report inconsistency flagged."),
            })

        timeline_status = (page_data.timeline_stage_assessment or {}).get("status")
        if timeline_status in {"too_old", "episode_mismatch", "timing_mismatch"}:
            flags.append({
                "package_code": page_data.package_code,
                "case_id": page_data.case_id,
                "page_id": page_data.page_id,
                "flag_type": "timeline_stage_of_care",
                "message": (page_data.timeline_stage_assessment or {}).get("note", "Timeline/stage mismatch flagged."),
            })

        return flags

    def _default_case_level_summary(case_result: CaseResult) -> Dict[str, Any]:
        return {
            "package_code": case_result.package_code,
            "case_id": case_result.case_id,
            "num_pages": len(case_result.pages),
            "num_flags": len(case_result.reviewer_flags),
            "overall_note": "Package-specific scaffold summary generated by main runner. Replace placeholder pipeline functions for substantive outputs.",
        }

    discovered_files = _safe_call(discover_case_files, input_root)
    if not isinstance(discovered_files, list) or len(discovered_files) == 0:
        discovered_files = _fallback_discover_case_files(input_root)

    cases: Dict[Tuple[str, str], CaseResult] = {}

    for discovered_file in discovered_files:
        case_key = (discovered_file.package_code, discovered_file.case_id)
        case_result = cases.setdefault(
            case_key,
            CaseResult(package_code=discovered_file.package_code, case_id=discovered_file.case_id)
        )

        page_stubs = _safe_call(split_file_into_pages, discovered_file)
        if not isinstance(page_stubs, list) or len(page_stubs) == 0:
            page_stubs = _fallback_split_file_into_pages(discovered_file)

        for page_stub in page_stubs:
            page_stub.setdefault("package_code", discovered_file.package_code)
            page_stub.setdefault("case_id", discovered_file.case_id)
            page_stub.setdefault("source_file", discovered_file.file_name)
            page_stub.setdefault("file_kind", discovered_file.file_kind)
            page_stub.setdefault("page_number", 1)
            page_stub.setdefault("page_id", f"{discovered_file.package_code}::{discovered_file.case_id}::{discovered_file.file_name}::1")
            page_stub.setdefault("image_path", discovered_file.file_path if discovered_file.file_kind in {"medical_image", "pdf_document"} else None)
            page_stub.setdefault("file_path", discovered_file.file_path)

            page_data = _safe_call(initialize_page_data, page_stub)
            if not isinstance(page_data, PageData):
                page_data = _fallback_initialize_page_data(page_stub)

            extracted_text = _safe_call(extract_text_for_page, page_data)
            if isinstance(extracted_text, str):
                writer_result = _safe_call(attach_text_to_page_data, page_data, extracted_text)
                if isinstance(writer_result, PageData):
                    page_data = writer_result
                else:
                    page_data.extracted_text = extracted_text
                    if page_data.file_kind == "written_report":
                        page_data.report_text = extracted_text

            quality_result = _safe_call(assess_image_quality, page_data)
            if isinstance(quality_result, dict):
                writer_result = _safe_call(write_image_quality_to_page, page_data, quality_result)
                if isinstance(writer_result, PageData):
                    page_data = writer_result
                else:
                    page_data.image_quality_assessment = quality_result

            image_result = _safe_call(interpret_medical_image, page_data)
            if isinstance(image_result, dict):
                writer_result = _safe_call(write_image_observations_to_page, page_data, image_result)
                if isinstance(writer_result, PageData):
                    page_data = writer_result
                else:
                    page_data.image_observations = image_result

            report_result = _safe_call(analyze_written_report, page_data)
            consistency_result = _safe_call(check_internal_report_consistency, page_data)
            if isinstance(report_result, dict) or isinstance(consistency_result, dict):
                writer_result = _safe_call(
                    write_report_analysis_to_page,
                    page_data,
                    report_result if isinstance(report_result, dict) else {},
                    consistency_result if isinstance(consistency_result, dict) else {},
                )
                if isinstance(writer_result, PageData):
                    page_data = writer_result
                else:
                    if isinstance(report_result, dict):
                        page_data.report_observations = report_result
                    if isinstance(consistency_result, dict):
                        page_data.internal_report_consistency = consistency_result

            correlation_result = _safe_call(correlate_image_with_report, page_data)
            if isinstance(correlation_result, dict):
                writer_result = _safe_call(write_correlation_to_page, page_data, correlation_result)
                if isinstance(writer_result, PageData):
                    page_data = writer_result
                else:
                    page_data.image_report_correlation = correlation_result

            stg_result = _safe_call(assess_package_and_stg_alignment, page_data)
            timeline_result = _safe_call(assess_timeline_and_stage_of_care, page_data)
            if isinstance(stg_result, dict) or isinstance(timeline_result, dict):
                writer_result = _safe_call(
                    write_context_assessments_to_page,
                    page_data,
                    stg_result if isinstance(stg_result, dict) else {},
                    timeline_result if isinstance(timeline_result, dict) else {},
                )
                if isinstance(writer_result, PageData):
                    page_data = writer_result
                else:
                    if isinstance(stg_result, dict):
                        page_data.package_stg_alignment = stg_result
                    if isinstance(timeline_result, dict):
                        page_data.timeline_stage_assessment = timeline_result

            reviewer_flags = _safe_call(build_reviewer_flags, page_data)
            if not isinstance(reviewer_flags, list):
                reviewer_flags = _default_reviewer_flags(page_data)

            case_result.pages.append(page_data)
            case_result.reviewer_flags.extend(reviewer_flags)

    run_outputs = RunOutputs(
        all_case_results=list(cases.values()),
        package_json_rows={pkg: [] for pkg in PACKAGE_OUTPUT_FIELDS.keys()},
    )

    for case_result in run_outputs.all_case_results:
        structured_row = build_package_json_row(case_result)
        case_result.structured_output_row = structured_row
        case_result.textual_summary = _extract_summary_pdf(structured_row)
        case_summary = _safe_call(build_case_level_summary, case_result)
        if not isinstance(case_summary, dict):
            case_summary = _default_case_level_summary(case_result)
        case_result.final_case_summary = case_summary

        run_outputs.package_json_rows.setdefault(case_result.package_code, []).append(structured_row)
        run_outputs.final_reviewer_flags.extend(case_result.reviewer_flags)
        run_outputs.final_case_summaries.append(case_result.final_case_summary)

    return run_outputs


In [ ]:

# =========================
# DISPLAY / FINAL OUTPUT SECTION
# =========================

def display_final_outputs(run_outputs: RunOutputs) -> None:
    """
    Display final results at the very end of the notebook.
    """

    print("=" * 80)
    print("PS2 PIPELINE COMPLETE — PACKAGE-SPECIFIC JSON OUTPUTS")
    print("=" * 80)
    print(f"Cases processed: {len(run_outputs.all_case_results)}")
    print(f"Reviewer flags: {len(run_outputs.final_reviewer_flags)}")
    print(f"Case summaries: {len(run_outputs.final_case_summaries)}")
    if run_outputs.written_files:
        print("Written files:")
        print(json.dumps(ensure_json_serializable(run_outputs.written_files), indent=2, ensure_ascii=False))

    for package_code in sorted(run_outputs.package_json_rows.keys()):
        rows = run_outputs.package_json_rows.get(package_code, [])
        print("\n" + "=" * 80)
        print(f"PACKAGE {package_code} — STRUCTURED JSON ROWS")
        print("=" * 80)
        if not rows:
            print("No rows available.")
        else:
            print(json.dumps(ensure_json_serializable(rows), indent=2, ensure_ascii=False))

    print("\n" + "=" * 80)
    print("FINAL REVIEWER FLAGS")
    print("=" * 80)
    print(json.dumps(ensure_json_serializable(run_outputs.final_reviewer_flags), indent=2, ensure_ascii=False))

    print("\n" + "=" * 80)
    print("FINAL CASE SUMMARIES")
    print("=" * 80)
    print(json.dumps(ensure_json_serializable(run_outputs.final_case_summaries), indent=2, ensure_ascii=False))


In [ ]:

# =========================
# EXECUTION CELL
# =========================

# Participants are expected to implement the pipeline stage functions above.
# Once implemented, the following is the intended end-of-notebook flow.

run_outputs = run_ps2_pipeline(INPUT_ROOT)
write_run_outputs(run_outputs, OUTPUT_ROOT)
display_final_outputs(run_outputs)
